# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will retrieve all record sets and display their `@id`, `name`, and fields.

In [ ]:
# List the available record sets and their fields
print("Available record sets:\n---------------------")
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'N/A')}")
    # List fields within the record set by @id
    print("  Fields:")
    for field in record_set['fields']:
        print(f"    - @id: {field['@id']}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this dataset, the main survey responses are in the record set with `@id` `cr:RecordSet_Main`. Let's extract records from all record sets.

In [ ]:
# List all available record_set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record set ids: {record_set_ids}\n")

# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    # Records are yielded as dicts keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(f"First records of {record_set_id}:")
    display(df.head())

# Choose primary record set for following analysis
main_record_set_id = record_set_ids[0]  # Usually the first one is main
print(f"Primary record set used for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field (by its @id) for analysis
# From the metadata and columns, let's use GAD-7 score field, which might have @id='cr:Field_gad7_score' or similar
main_df = dataframes[main_record_set_id]

# Guess the GAD-7 field (replace with actual if known exactly)
numeric_field = [col for col in main_df.columns if 'gad7' in col.lower() or 'GAD7' in col][0]
print(f"Selected numeric field: {numeric_field}\n")

threshold = 10
filtered_df = main_df[main_df[numeric_field].astype(float) > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g., 'cr:Field_gender' if present
group_field_candidates = [col for col in main_df.columns if 'gender' in col.lower() or 'sex' in col.lower()]
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped average {numeric_field} by {group_field}:")
    display(grouped_df)
else:
    print("No clear categorical field (gender/sex) for grouping found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field].astype(float).dropna(), kde=True, bins=20, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If grouping by gender field exists, visualize by group
if group_field_candidates:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the FAIR² Kilifi County Mental Health Survey data via the Croissant schema and explored its structure.
- Performed numeric and group-based analysis on GAD-7 scores, identifying participants with scores above 10 and analyzing distributions by gender if available.
- The dataset supports further exploration, including analysis of other mental health indicators (e.g., PHQ-9, PSQ), cross-demographic trends, or temporal analyses.

**Next steps:** Try extending the notebook by exploring missing data patterns, survey completion rates, or correlating scores between different assessment tools!